# Tráfico en Madrid: Evaluación Final y Demo de Predicción

Hemos llegado al final del proyecto. Hasta aquí, todos los pasos —limpieza, feature engineering, selección de modelos y optimización— se han ejecutado sin abrir el test set en ningún momento. Los datos de test se apartaron al inicio y el modelo nunca los ha visto. Este notebook los abre por primera vez.

Lo que hacemos aquí es simple: cargamos el Random Forest que guardó el NB03, lo ejecutamos sobre el test set y comprobamos si las métricas de validación se sostienen con datos completamente nuevos. Después analizamos dónde falla el modelo, vemos qué variables considera más relevantes, explicamos por qué toma una decisión concreta, y cerramos con una función de predicción que responde la pregunta práctica del proyecto: dado un punto de medida de Madrid y un momento del día, ¿habrá atasco?

In [1]:
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

Constantes, rutas y configuración estética. Las mismas paletas y ajustes que en los notebooks anteriores para que las gráficas sean coherentes.

In [2]:
RANDOM_STATE = 42
DATA_FINAL = Path("../data/final")
DATA_PROC = Path("../data/processed")

RUTA_MODELO = DATA_FINAL / "modelo_congestion.joblib"
RUTA_METADATOS = DATA_FINAL / "modelo_metadatos.json"

FEATURES_MODELO = [
    "tipo_elem",
    "distrito",
    "utm_x",
    "utm_y",
    "hora",
    "dia_semana",
    "es_laborable",
    "es_hora_punta",
]

DISTRITOS_MADRID = {
    1: "Centro",
    2: "Arganzuela",
    3: "Retiro",
    4: "Salamanca",
    5: "Chamartín",
    6: "Tetuán",
    7: "Chamberí",
    8: "Fuencarral",
    9: "Moncloa",
    10: "Latina",
    11: "Carabanchel",
    12: "Usera",
    13: "Vallecas",
    14: "Moratalaz",
    15: "C. Lineal",
    16: "Hortaleza",
    17: "Villaverde",
    18: "V. Vallecas",
    19: "Vicálvaro",
    20: "San Blas",
    21: "Barajas",
}

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="ticks", palette="muted", font_scale=0.9)
plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.frameon": False,
        "axes.grid": True,
        "grid.linestyle": ":",
        "grid.alpha": 0.35,
    }
)

COLOR_PRIMARIO = "#4C72B0"
COLOR_SECUNDARIO = "#DD8452"

## 1. Carga del modelo y los datos

Recogemos lo que guardó el NB03 en disco: el modelo entrenado en formato joblib y los metadatos con las métricas de validación. También cargamos dos conjuntos de evaluación: el test principal y un segundo conjunto adicional. Ninguno de los dos ha sido abierto en ningún paso anterior.

In [ ]:
modelo = joblib.load(RUTA_MODELO)
with RUTA_METADATOS.open(encoding="utf-8") as f:
    meta = json.load(f)

print(f"Modelo: {type(modelo.named_steps['clf']).__name__}")
print(f"F1 en validación: {meta['f1_val']:.4f}")
print(f"Features: {meta['features']}")

In [ ]:
def cargar_split(nombre):
    X = pd.read_parquet(DATA_FINAL / f"X_{nombre}.parquet").drop(columns=["id"])
    y = pd.read_parquet(DATA_FINAL / f"y_{nombre}.parquet").squeeze()
    return X[FEATURES_MODELO], y


X_test, y_test = cargar_split("test")
X_eval, y_eval = cargar_split("eval_final")

X_test_raw = pd.read_parquet(DATA_FINAL / "X_test.parquet")
df_sensores = pd.read_parquet(DATA_PROC / "sensores/sensores.parquet")

print(f"Test:       {X_test.shape[0]:,} filas × {X_test.shape[1]} features")
print(f"Eval final: {X_eval.shape[0]:,} filas × {X_eval.shape[1]} features")
print(f"Congestionado en test: {y_test.mean():.2%} de los registros")

## 2. Evaluación en el test set

El momento que llevábamos evitando desde el principio del proyecto. El test set tiene 632.603 registros, de los que el 6.9% son congestionados. Ejecutamos el modelo y calculamos todas las métricas. Necesitamos tanto las predicciones binarias como las probabilidades: las primeras dan precision, recall y F1; las segundas, las curvas ROC y precision-recall.

In [ ]:
y_pred = modelo.predict(X_test)
y_pred_prob = modelo.predict_proba(X_test)[:, 1]

metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "pr_auc": average_precision_score(y_test, y_pred_prob),
    "roc_auc": roc_auc_score(y_test, y_pred_prob),
}
pd.DataFrame([metricas_test], index=["Test"]).T

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Fluido", "Congestionado"]))

La clase "Fluido" tiene un F1 de 0.98 porque es la mayoría y la más fácil. Lo que importa es "Congestionado": F1 de 0.78 con precision 0.73 y recall 0.82 confirma que el modelo detecta 8 de cada 10 atascos reales y que 3 de cada 4 alertas que lanza son ciertas.

### Comparativa validación → test

Una caída grande entre las métricas de validación y test sería señal de overfitting. Si quedan cerca, el modelo generaliza bien.

In [ ]:
comparativa = pd.DataFrame(
    {
        "Validación": [
            meta["f1_val"],
            meta["precision_val"],
            meta["recall_val"],
            meta["pr_auc_val"],
        ],
        "Test": [
            metricas_test["f1"],
            metricas_test["precision"],
            metricas_test["recall"],
            metricas_test["pr_auc"],
        ],
    },
    index=["F1", "Precision", "Recall", "PR-AUC"],
)
comparativa["Delta"] = comparativa["Test"] - comparativa["Validación"]
comparativa

Los deltas son prácticamente nulos: F1 baja solo 0.002 puntos (de 0.778 a 0.776). El modelo no ha memorizado el entrenamiento; lo que aprendió con 5 millones de registros funciona igual de bien con datos nuevos.

## 3. Segundo conjunto de evaluación

Aparte del test principal, cargamos un segundo conjunto adicional para confirmar que las cifras no dependen del split concreto. Si los dos conjuntos dan métricas similares, tenemos más confianza en la estabilidad del modelo.

In [ ]:
y_pred_ev = modelo.predict(X_eval)
y_pred_prob_ev = modelo.predict_proba(X_eval)[:, 1]

metricas_eval = {
    "accuracy": accuracy_score(y_eval, y_pred_ev),
    "precision": precision_score(y_eval, y_pred_ev),
    "recall": recall_score(y_eval, y_pred_ev),
    "f1": f1_score(y_eval, y_pred_ev),
    "pr_auc": average_precision_score(y_eval, y_pred_prob_ev),
    "roc_auc": roc_auc_score(y_eval, y_pred_prob_ev),
}
pd.DataFrame(
    {
        f"Test ({len(X_test) // 1000:,}K)": metricas_test,
        f"Eval final ({len(X_eval) // 1000:,}K)": metricas_eval,
    }
)

Las cifras son coherentes: F1 de 0.776 en test vs 0.789 en el conjunto adicional, ROC-AUC de 0.976 vs 0.969. Las diferencias son mínimas y van en ambas direcciones, lo que descarta que una de las dos sea mejor o peor por casualidad del split.

## 4. Matriz de confusión

La matriz descompone el error en cuatro tipos. Los falsos negativos son los atascos que nos perdemos —en un sistema de aviso son el error más caro porque el usuario no recibe la alerta—. Los falsos positivos son las falsas alarmas.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Fluido", "Congestionado"],
    yticklabels=["Fluido", "Congestionado"],
    ax=ax,
    linewidths=0.5,
    linecolor="white",
)
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusión — test set")
plt.tight_layout()
plt.show()

In [ ]:
tn, fp, fn, tp = cm.ravel()
total_cong = tp + fn

print(f"Verdaderos positivos (TP):  {tp:>10,}   ← atascos detectados")
print(f"Falsos negativos (FN):      {fn:>10,}   ← atascos no detectados")
print(f"Falsos positivos (FP):      {fp:>10,}   ← alertas falsas")
print(f"Verdaderos negativos (TN):  {tn:>10,}   ← fluido bien identificado")
print()
print(f"Tasa de detección (recall):       {tp / total_cong:.1%}")
print(f"Fiabilidad de alertas (precision): {tp / (tp + fp):.1%}")

De los 43.807 atascos reales en el test set, el modelo detecta 36.070 (82.3%) y se pierde 7.737 (17.7%). Por cada aviso que lanza, acierta el 73.4% de las veces —las otras 13.082 alertas son falsas alarmas. La relación es razonable para un sistema de aviso: preferimos lanzar algunas alertas de más a perdernos atascos reales, y ese balance es ajustable subiendo el umbral de probabilidad.

## 5. Curvas ROC y precision-recall

La curva ROC mueve el umbral de decisión de 0 a 1 y traza la tasa de verdaderos positivos frente a la de falsos positivos. El AUC resume la calidad del modelo independientemente del umbral: 0.5 sería un clasificador aleatorio, 1.0 sería perfecto. La curva precision-recall es más informativa cuando hay desbalance porque no incluye los verdaderos negativos en el cálculo.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_roc = roc_auc_score(y_test, y_pred_prob)

prec_c, rec_c, _ = precision_recall_curve(y_test, y_pred_prob)
ap = average_precision_score(y_test, y_pred_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(fpr, tpr, color=COLOR_PRIMARIO, lw=2, label=f"AUC = {auc_roc:.3f}")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Azar (AUC = 0.5)")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curva ROC")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax = axes[1]
ax.plot(rec_c, prec_c, color=COLOR_PRIMARIO, lw=2, label=f"AP = {ap:.3f}")
ax.axhline(
    y_test.mean(), color="gray", lw=1, linestyle="--", label=f"Azar (AP = {y_test.mean():.3f})"
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva precision-recall")
ax.legend(loc="upper right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

Un ROC-AUC de 0.976 es muy alto: el modelo separa bien las dos clases para prácticamente cualquier umbral. La curva precision-recall (AP = 0.810) queda muy por encima de la línea base del clasificador aleatorio, que con 6.9% de positivos tendría AP ≈ 0.069. Multiplicar por doce el Average Precision de un clasificador aleatorio con este nivel de desbalance es un resultado sólido.

## 6. Importancia de variables

Miramos qué variables considera más relevantes el modelo. Tiene dos utilidades: validar que aprendió cosas razonables y entender qué factores conducen las predicciones. El Random Forest mide la importancia de cada variable por cuánto reduce la impureza en los splits donde se usa, promediado sobre todos los árboles del bosque.

In [ ]:
importancias = pd.Series(
    modelo.named_steps["clf"].feature_importances_,
    index=FEATURES_MODELO,
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(importancias.index, importancias.values, color=COLOR_PRIMARIO)
ax.set_xlabel("Importancia relativa")
ax.set_title("Importancia de variables — Random Forest")
plt.tight_layout()
plt.show()

importancias.sort_values(ascending=False)

El resultado sorprende un poco. Las tres variables más importantes son las **coordenadas geográficas** (utm_y 27.8%, utm_x 26.2%) y la **hora** (26.7%), que suman el 80% del peso total. El día de la semana (6.5%) queda en cuarto lugar, y el distrito apenas llega al 4.2%.

Esto tiene sentido una vez se piensa un momento: las coordenadas codifican la ubicación con mucha más precisión que el código de distrito, que agrupa zonas muy heterogéneas. El modelo está usando esencialmente GPS + hora como señal principal, y con eso puede mapear la ciudad en zonas de alta y baja congestión aprendidas de los datos. Las variables derivadas —`es_hora_punta` y `es_laborable`— tienen poco peso porque el modelo ya puede inferir esa información directamente de `hora` y `dia_semana`, pero siguen contribuyendo algo al margen.

### ¿Por qué predice congestión? Análisis de sensibilidad

Las importancias globales dicen cuánto usa el modelo cada variable en promedio sobre todo el test set, pero no dicen qué peso tiene cada factor en una predicción concreta. Para eso, tomamos un caso real —la vía de servicio de la M-30 junto a la Avenida de la Albufera, un lunes a las 8h (predicción: 85% de congestión)— y medimos cuánto cambia la predicción si modificamos cada factor temporalmente.

In [ ]:
ID_EXPL = 5824  # Vía Servicio M-30 - M-30-Av. Albufera
nombre_expl = df_sensores.loc[df_sensores["id"] == ID_EXPL, "nombre"].iloc[0]

fila_base = X_test_raw[X_test_raw["id"] == ID_EXPL].iloc[0][FEATURES_MODELO].copy().astype(float)
fila_base["hora"] = 8.0
fila_base["dia_semana"] = 0.0
fila_base["es_laborable"] = 1.0
fila_base["es_hora_punta"] = 1.0
prob_base = modelo.predict_proba(fila_base.to_frame().T)[0, 1]


def perturb(cambios):
    f = fila_base.copy()
    for k, v in cambios.items():
        f[k] = v
    return modelo.predict_proba(f.to_frame().T)[0, 1]


escenarios_cf = {
    "hora 8h → 3h (madrugada)": {"hora": 3.0, "es_hora_punta": 0.0},
    "hora 8h → 14h (mediodía)": {"hora": 14.0, "es_hora_punta": 0.0},
    "hora 8h → 18h (tarde)": {"hora": 18.0, "es_hora_punta": 1.0},
    "lunes → domingo": {"dia_semana": 6.0, "es_laborable": 0.0, "es_hora_punta": 0.0},
    "fuera de hora punta": {"es_hora_punta": 0.0},
    "día no laborable": {"es_laborable": 0.0, "es_hora_punta": 0.0},
}

probs_cf = {desc: perturb(c) for desc, c in escenarios_cf.items()}
delta_cf = pd.Series({desc: p - prob_base for desc, p in probs_cf.items()}).sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = [COLOR_SECUNDARIO if v < 0 else COLOR_PRIMARIO for v in delta_cf.values]
ax.barh(delta_cf.index, delta_cf.values, color=colors, height=0.6)
ax.axvline(0, color="black", lw=0.8)

for i, (desc, delta) in enumerate(delta_cf.items()):
    prob_alt = delta + prob_base
    offset = 0.012 if delta >= 0 else -0.012
    ha = "left" if delta >= 0 else "right"
    ax.text(delta + offset, i, f"{prob_alt:.0%}", va="center", ha=ha, fontsize=9)

ax.set_xlabel(f"Cambio en probabilidad de congestión (base: {prob_base:.0%})")
ax.set_title(f"¿Qué cambiaría la predicción? — {nombre_expl[:48]}, lunes 08:00")
plt.tight_layout()
plt.show()

El gráfico deja ver en qué se fija el modelo para este caso concreto. Mover la consulta a las 3 de la madrugada o al domingo baja la predicción de 85% a 0%: hora y día de la semana son los factores más decisivos. Lo opuesto también es revelador: preguntar por las 14h o las 18h sube la probabilidad a 100%, lo que confirma que esta vía de servicio de la M-30 está saturada prácticamente todo el día en laborables, no solo en la hora punta de la mañana.

El modelo no está usando ningún atajo raro: aprendió el patrón real del tráfico. Esta zona (acceso a la M-30 por Avenida de la Albufera) es uno de los puntos con mayor congestión sostenida de la ciudad, y el modelo lo capturó correctamente.

### Camino de decisión en un árbol

Las importancias globales y el análisis de sensibilidad muestran qué variables pesan más en promedio, pero no el razonamiento nodo a nodo. En un Random Forest podemos extraer el camino exacto que recorrió uno de los 100 árboles para llegar a su predicción: qué variable comprobó en cada nodo, el umbral que usó y hacia qué rama fue.

In [ ]:
tree = modelo.named_steps["clf"].estimators_[0]
fila_expl = fila_base.to_frame().T

node_indicator = tree.decision_path(fila_expl)
leaf_id = tree.apply(fila_expl)[0]
node_ids = node_indicator.indices[node_indicator.indptr[0] : node_indicator.indptr[1]]

MAX_SPLITS = 10

print(f"Árbol 0 — {nombre_expl[:55]}, lunes 08:00")
print(f"Profundidad del árbol: {tree.get_depth()}  |  Nodos en el camino: {len(node_ids)}")
print(f"\n{'Nodo':>5}  {'Variable':<18}  {'Umbral':>10}  {'Muestra':>10}  {'Rama'}")
print("─" * 62)

for node_id in node_ids[:MAX_SPLITS]:
    if node_id == leaf_id:
        v = tree.tree_.value[node_id][0]
        p = v[1] / v.sum()
        print(
            f"\n{'HOJA':>5}  → P(congestionado) = {p:.1%}"
            f"  ({int(v[1]):,} positivos / {int(v.sum()):,} muestras)"
        )
        break
    f_idx = tree.tree_.feature[node_id]
    t = tree.tree_.threshold[node_id]
    val_m = fila_expl.iloc[0, f_idx]
    rama = "← ≤" if val_m <= t else "→ >"
    print(f"{node_id:>5}  {FEATURES_MODELO[f_idx]:<18}  {t:>10.3f}  {val_m:>10.3f}  {rama}")

if len(node_ids) > MAX_SPLITS and node_ids[MAX_SPLITS - 1] != leaf_id:
    print(f"\n  ··· {len(node_ids) - MAX_SPLITS} nodos más hasta la hoja")

## 7. Análisis de errores

Las métricas globales no dicen cuándo ni dónde falla el modelo. Los errores rara vez están distribuidos de forma uniforme: suelen concentrarse en situaciones concretas, y esas concentraciones son la señal más útil para entender las limitaciones del sistema.

In [ ]:
errores = X_test.copy()
errores["y_real"] = y_test.values
errores["y_pred"] = y_pred
errores["acierto"] = (errores["y_real"] == errores["y_pred"]).astype(int)

print(f"Tasa de error global en test: {1 - errores['acierto'].mean():.2%}")

### Errores por hora del día

Lo más intuitivo es mirar si los errores se concentran en ciertas horas. La tasa de error global es 3.29%, pero ese promedio esconde variaciones muy grandes a lo largo del día.

In [ ]:
tasa_error_hora = 1 - errores.groupby("hora")["acierto"].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(
    tasa_error_hora.index, tasa_error_hora.values, marker="o", color=COLOR_SECUNDARIO, linewidth=1.5
)
ax.axhline(
    1 - errores["acierto"].mean(), color="gray", lw=1, linestyle="--", label="Media global (3.29%)"
)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Tasa de error")
ax.set_title("Tasa de error por hora del día")
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

La diferencia es enorme. A las 3-4 de la madrugada el error es inferior al 0.3%; a las 9h y las 18h supera el 6%, más de veinte veces la tasa de madrugada. Los dos picos coinciden exactamente con las horas de transición de las puntas de mañana y tarde: el tráfico está cambiando de estado en esas franjas y la misma hora puede corresponder a situaciones muy distintas dependiendo del día concreto, del clima o de un incidente. Sin información en tiempo real, el modelo no puede distinguirlos.

### Errores por distrito

Hay también diferencias espaciales. Los distritos con más actividad y variabilidad en el tráfico son los más difíciles de predecir.

In [ ]:
tasa_error_distrito = (1 - errores.groupby("distrito")["acierto"].mean()).sort_values(
    ascending=True
)

labels = [DISTRITOS_MADRID.get(int(d), str(d)) for d in tasa_error_distrito.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, tasa_error_distrito.values, color=COLOR_SECUNDARIO)
ax.axvline(
    1 - errores["acierto"].mean(), color="gray", lw=1, linestyle="--", label="Media global (3.29%)"
)
ax.set_xlabel("Tasa de error")
ax.set_title("Tasa de error por distrito")
ax.legend()
plt.tight_layout()
plt.show()

El patrón es el esperado. Los tres distritos con más error son Centro (7.0%), Chamberí (5.8%) y Salamanca (5.3%): los más densos de la ciudad, donde la frontera entre fluido y congestionado es la más difusa y variable de un día para otro. En el extremo opuesto, Latina (1.4%), Tetuán (1.5%) y Villaverde (1.6%) tienen pocos atascos registrados y el modelo casi siempre predice fluido acertando, aunque no porque aprenda nada especialmente interesante.

## 8. Demo: predicción en tiempo real

La función que viene a continuación responde la pregunta práctica del proyecto: dado un punto de medida de Madrid y un momento concreto, ¿habrá atasco? Para elegir el sensor, buscamos por nombre de calle en el catálogo.

In [ ]:
for calle in ["Castellana", "Gran Via", "M-30", "Alcal", "Recoletos"]:
    hits = df_sensores[df_sensores["nombre"].str.contains(calle, case=False, na=False)][
        ["id", "nombre", "distrito"]
    ].head(3)
    if not hits.empty:
        print(f"--- {calle} ---")
        print(hits.to_string(index=False))
        print()

In [ ]:
ID_CASTELLANA = 3852  # Paseo Castellana S-N - Pl. Colon-Ayala
ID_GRAN_VIA = 3672  # Gran Vía 52 S-N
ID_M30 = 5824  # Vía Servicio M-30 - M-30-Av. Albufera
ID_ALCALA = 3645  # Alcalá - Argentina - Concejal Julio Gómez
ID_RECOLETOS = 4244  # Paseo Recoletos N-S - Prim-Pl. Cibeles

La función recoge las características del sensor directamente de los datos de test, donde ya están en el formato que espera el modelo. Solo sobreescribe las variables temporales: hora, día de la semana y los dos indicadores derivados que se calculan a partir de ellos.

In [ ]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]


def predict_congestion(id_sensor: int, hora: int, dia_semana: int) -> dict:
    """Predice congestión en un sensor dado una hora y día de la semana."""
    fila_base = X_test_raw[X_test_raw["id"] == id_sensor]
    if fila_base.empty:
        raise ValueError(f"Sensor {id_sensor} no encontrado en los datos")
    fila = fila_base.iloc[0][FEATURES_MODELO].copy().astype(float)
    fila["hora"] = float(hora)
    fila["dia_semana"] = float(dia_semana)
    fila["es_laborable"] = float(dia_semana < 5)
    fila["es_hora_punta"] = float(dia_semana < 5 and (7 <= hora <= 9 or 17 <= hora <= 20))
    prob = modelo.predict_proba(fila.to_frame().T)[0, 1]
    nombre = df_sensores.loc[df_sensores["id"] == id_sensor, "nombre"]
    return {
        "sensor": nombre.iloc[0] if not nombre.empty else str(id_sensor),
        "estado": "CONGESTIONADO" if prob >= 0.5 else "FLUIDO",
        "probabilidad": prob,
    }

Cinco escenarios elegidos para mostrar el rango completo de situaciones. La M-30 aparece dos veces a propósito: un lunes a las 8h (hora punta) y un domingo a las 3h (madrugada). Si el modelo estuviera memorizando el sensor en vez de aprender el patrón temporal, los dos darían el mismo resultado.

In [ ]:
escenarios = [
    (ID_M30, 8, 0, "M-30 Albufera — lunes, hora punta mañana"),
    (ID_GRAN_VIA, 19, 4, "Gran Vía — viernes, hora punta tarde"),
    (ID_ALCALA, 8, 0, "Alcalá — lunes, hora punta mañana"),
    (ID_M30, 3, 6, "M-30 Albufera — domingo, madrugada"),
    (ID_CASTELLANA, 11, 6, "Castellana — domingo, mañana"),
]

filas = []
for id_sensor, hora, dow, descripcion in escenarios:
    r = predict_congestion(id_sensor, hora, dow)
    filas.append(
        {
            "Escenario": descripcion,
            "Hora": f"{hora:02d}:00",
            "Predicción": r["estado"],
            "Probabilidad": f"{r['probabilidad']:.0%}",
        }
    )

pd.DataFrame(filas)

La M-30 un lunes a las 8h da 85% y el domingo a las 3h da 0%: el mismo sensor, resultados opuestos. Esto confirma que el modelo aprendió el patrón temporal, no el sensor. Gran Vía a las 19h del viernes alcanza el 97% —es uno de los puntos más saturados de Madrid en esa franja—, mientras que la Castellana el domingo por la mañana queda prácticamente vacía.

### Perfil de probabilidad a lo largo del día — M-30 Albufera

La curva siguiente muestra cómo evoluciona la probabilidad de congestión hora a hora en la vía de servicio de la M-30 junto a la Avenida de la Albufera, comparando un lunes laborable con un domingo. Es la vista que mostraría una aplicación real de monitoreo de tráfico.

In [ ]:
horas = list(range(24))
prob_lab = [predict_congestion(ID_M30, h, 0)["probabilidad"] for h in horas]
prob_dom = [predict_congestion(ID_M30, h, 6)["probabilidad"] for h in horas]
nombre_viz = df_sensores.loc[df_sensores["id"] == ID_M30, "nombre"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horas, prob_lab, marker="o", color=COLOR_PRIMARIO, linewidth=1.8, label="Lunes (laborable)")
ax.plot(horas, prob_dom, marker="o", color=COLOR_SECUNDARIO, linewidth=1.8, label="Domingo")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Umbral de decisión (0.5)")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Probabilidad de congestión")
ax.set_title(f"Perfil diario — {nombre_viz[:60]}")
ax.set_xticks(range(0, 24, 2))
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

El lunes muestra un patrón de saturación casi total: la probabilidad sube rápidamente a partir de las 8h, alcanza el 98-100% desde las 10h hasta las 19h, y cae en picado después de las 21h. Esta vía de servicio de la M-30 no tiene punta doble —está congestionada todo el día laborable— lo que explica por qué el análisis de sensibilidad también mostraba que las 14h eran tan malas como las 8h.

El domingo tiene un perfil diferente: mañana tranquila hasta las 10h y luego un pico en la tarde (13-14h, probable actividad en las zonas comerciales de la Avenida de la Albufera). Hay algo que llama la atención aquí: la tarde del domingo en esta zona tiene congestion no despreciable, algo que la intuición general no anticiparía pero que el modelo aprendió de los datos reales.

## 9. Conclusiones

Hemos construido un pipeline de principio a fin para predecir congestión en los puntos de medida del Ayuntamiento de Madrid. Los resultados en test confirman lo que ya indicaba la validación cruzada: el modelo generaliza bien y las métricas son consistentes entre los dos conjuntos de evaluación.

**Qué funciona.** El Random Forest alcanza F1 = 0.776 en test (precision 73.4%, recall 82.3%, ROC-AUC 0.976) con solo ocho variables. De los 43.807 atascos reales en el test set, detecta 36.070 —8 de cada 10. Lo más revelador del análisis de importancias es que las coordenadas geográficas (utm_y + utm_x, 54% del peso total) dominan sobre la hora (27%), y el distrito queda casi irrelevante (4.2%): el modelo aprendió a mapear la ciudad en zonas de congestión a nivel de coordenadas, con una resolución mucho mayor que el código de distrito.

**Dónde falla.** Los errores se concentran en las horas de transición —9h y 18h con tasas superiores al 6%— y en los distritos más densos (Centro, Chamberí, Salamanca). Son los casos donde factores no medidos como un accidente, la lluvia o un evento deportivo marcan la diferencia entre fluido y congestionado. Sin esa información en tiempo real, no hay forma de distinguirlos.

**Posibles mejoras.** Añadir más histórico permitiría capturar patrones estacionales que con los datos actuales no son visibles. Cruzar con datos meteorológicos de AEMET explicaría parte de los errores de transición. Incluir la carga del sensor en la hora anterior convertiría el sistema en un modelo de serie temporal, con un salto de calidad importante. El umbral de decisión es configurable: bajándolo se recuperan más atascos a costa de más falsas alarmas, dependiendo del caso de uso concreto.